In [2]:
# imports
import pandas as pd
from pathlib import Path

In [3]:
# categorizing data (nominal, ordinal, dichotomous, discrete, ratio, interval)
nominal_cols = ["Season", "DayNum", "WTeamID", "LTeamID", "WLoc", "NumOT"]
ordinal_cols = []
dichotomous_cols = []
discrete_cols = ['WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WOR', 'WDR', 'WAst', 'WTO', 'WStl', 'WBlk',
                 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LOR', 'LDR', 'LAst', 'LTO', 'LStl', 'LBlk']
ratio_cols = ["WScore", "LScore"]
interval_cols = []

In [4]:
root = Path("../data")
W_files = []
M_files = []

for file in root.rglob("*.csv"):
    name = file.name.capitalize()

    if name.startswith("W"):
        W_files.append(file)
    elif name.startswith("M"):
        M_files.append(file)

print(f"Women: {len(W_files)}, Men: {len(M_files)}")



Women: 14, Men: 17


In [5]:
# Group 1: Identity tables(joins keys, not features)
identity_table_file = ["Mteams", "MSeasons", "MTeamConferences", "Cities"]
# Group 2: Game Results
game_results = [
    "MRegularSeasonCompactResults", "MRegularSeasonDetailedResults", 
    "MNCAATourneyCompactResults", "MNCAATourneyDetailedResults",
]
# Group 3: Tournament Structure
tournament_structure = ["MNCAATourneySeeds", "MNCAATourneySlots", "MNCAATourneySeedRoundSlots",]
# Group 4: External Rankings
external_ranking = ["MMasseyOrdinals"]
# Group 5: Geography/Misc
geography = ["MGameCities", "MTeamCoaches", "MConferenceTourneyGames", "Cities"]

In [6]:
df = pd.read_csv("../data/MMasseyOrdinals.csv")
print(df.columns.tolist())
print(df.shape)
df.head()

['Season', 'RankingDayNum', 'SystemName', 'TeamID', 'OrdinalRank']
(5865001, 5)


,Season,RankingDayNum,SystemName,TeamID,OrdinalRank
0,2003,35,SEL,1102,159
1,2003,35,SEL,1103,229
2,2003,35,SEL,1104,12
3,2003,35,SEL,1105,314
4,2003,35,SEL,1106,260


In [7]:
for col in df.columns:
    print(df[col].nunique())

24
91
197
372
365


In [8]:
df['RankingDayNum'].value_counts().sort_index()

RankingDayNum
0         345
1        1066
2        6644
6         736
7        2853
        ...  
125       100
126     43115
127     68104
128    311829
133    402089
Name: count, Length: 91, dtype: int64

In [9]:
filtered = df[df['RankingDayNum'] == 133]
print(filtered['SystemName'].value_counts().head(10))

SystemName
MOR    7992
POM    7989
BIH    7645
WLK    7624
DOL    7623
COL    7622
DOK    7009
WIL    7004
RPI    6927
RTH    6918
Name: count, dtype: int64


In [10]:
# Group 2 - Game Results
tourney_results = pd.read_csv('../data/MNCAATourneyCompactResults.csv')
regular_season_detailed = pd.read_csv('../data/MRegularSeasonDetailedResults.csv')

# Group 3 - Tournament Structure
tourney_seeds = pd.read_csv('../data/MNCAATourneySeeds.csv')

# Group 4 - External Rankings
massey = pd.read_csv('../data/MMasseyOrdinals.csv')

print("tourney_results:", tourney_results.shape)
print("regular_season_detailed:", regular_season_detailed.shape)
print("tourney_seeds:", tourney_seeds.shape)
print("massey:", massey.shape)

tourney_results: (2585, 8)
regular_season_detailed: (124529, 34)
tourney_seeds: (2694, 3)
massey: (5865001, 5)


In [11]:
# Season range
TRAIN_START = 2010
VALID_SEASON = 2025
TEST_SEASON = 2026

# Filter all files to 2010 onwards
tourney_results = tourney_results[tourney_results['Season'] >= TRAIN_START].reset_index(drop=True)
regular_season_detailed = regular_season_detailed[regular_season_detailed['Season'] >= TRAIN_START].reset_index(drop=True)
tourney_seeds = tourney_seeds[tourney_seeds['Season'] >= TRAIN_START].reset_index(drop=True)

# Filter Massey — final pre-tournament ranking + 2 systems only
systems_to_keep = ['MOR', 'POM']
massey_filtered = massey[
    (massey['RankingDayNum'] == 133) &
    (massey['SystemName'].isin(systems_to_keep)) &
    (massey['Season'] >= TRAIN_START)
].reset_index(drop=True)

print("tourney_results:", tourney_results.shape)
print("regular_season_detailed:", regular_season_detailed.shape)
print("tourney_seeds:", tourney_seeds.shape)
print("massey_filtered:", massey_filtered.shape)

tourney_results: (1001, 8)
regular_season_detailed: (90455, 34)
tourney_seeds: (1085, 3)
massey_filtered: (11302, 5)


In [12]:
print(massey_filtered.head())
print(tourney_results.head())
print(regular_season_detailed.head())
print("Tourney Seed \n")
print(tourney_seeds.head())

   Season  RankingDayNum SystemName  TeamID  OrdinalRank
0    2010            133        MOR    1102          227
1    2010            133        MOR    1103          118
2    2010            133        MOR    1104           57
3    2010            133        MOR    1105          337
4    2010            133        MOR    1106          303
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    2010     134     1115      61     1457      44    N      0
1    2010     136     1124      68     1358      59    N      0
2    2010     136     1139      77     1431      59    N      0
3    2010     136     1140      99     1196      92    N      2
4    2010     136     1242      90     1250      74    N      0
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT  WFGM  WFGA  \
0    2010       7     1143      75     1293      70    H      0    24    52   
1    2010       7     1314      88     1198      72    H      0    34    61   
2    2010       7     1326     100   

In [13]:
# Extract numeric seed from string like 'W01', 'X16', 'Y04'
tourney_seeds['SeedNum'] = tourney_seeds['Seed'].str.extract(r'(\d+)').astype(int)

# Drop original Seed string column
tourney_seeds = tourney_seeds.drop(columns=['Seed'])

print(tourney_seeds.head(10))

   Season  TeamID  SeedNum
0    2010    1246        1
1    2010    1452        2
2    2010    1307        3
3    2010    1458        4
4    2010    1396        5
5    2010    1266        6
6    2010    1155        7
7    2010    1400        8
8    2010    1448        9
9    2010    1281       10


In [14]:
massey_wide = massey_filtered.pivot_table(
    index=['Season', 'TeamID'],
    columns='SystemName',
    values='OrdinalRank'
).reset_index()
massey_wide = massey_wide.rename(columns={
    'MOR': 'MOR_Rank',
    'POM': 'POM_Rank'
})


print(massey_wide.shape)
print(massey_wide.head())
print(massey_wide.isnull().sum())

(5651, 4)
SystemName  Season  TeamID  MOR_Rank  POM_Rank
0             2010    1102     227.0     246.0
1             2010    1103     118.0     119.0
2             2010    1104      57.0      64.0
3             2010    1105     337.0     336.0
4             2010    1106     303.0     302.0
SystemName
Season      0
TeamID      0
MOR_Rank    0
POM_Rank    0
dtype: int64


In [15]:
win_rows = regular_season_detailed[[
    'Season', 'WTeamID', 'WScore', 'LScore',
    'WFGM', 'WFGA', 'WFGM3', 'WFTM', 'WFTA',
    'WOR', 'WDR', 'WTO'
]].rename(columns={
    'WTeamID': 'TeamID',
    'WScore': 'PointsFor',
    'LScore' : 'PointsAgainst',
    'WFGM': 'FGM', 'WFGA': 'FGA', 'WFGM3':'FGM3',
    'WFTM':'FTM', 'WFTA':'FTA',
    'WOR':'OR', 'WDR': 'DR', 'WTO':'TO'
})
win_rows['Win']=1

# Loser rows
loss_rows = regular_season_detailed[[
    'Season', 'LTeamID', 'LScore', 'WScore',
    'LFGM', 'LFGA', 'LFGM3', 'LFTM', 'LFTA',
    'LOR', 'LDR', 'LTO'
]].rename(columns={
    'LTeamID': 'TeamID',
    'LScore': 'PointsFor',
    'WScore': 'PointsAgainst',
    'LFGM': 'FGM', 'LFGA': 'FGA', 'LFGM3': 'FGM3',
    'LFTM': 'FTM', 'LFTA': 'FTA',
    'LOR': 'OR', 'LDR': 'DR', 'LTO': 'TO'
})
loss_rows['Win'] = 0


# combine
team_games =pd.concat([win_rows, loss_rows], ignore_index=True)

print(team_games.shape)
print(team_games.head())

(180910, 13)
   Season  TeamID  PointsFor  PointsAgainst  FGM  FGA  FGM3  FTM  FTA  OR  DR  \
0    2010    1143         75             70   24   52     5   22   32  13  19   
1    2010    1314         88             72   34   61     4   16   19  12  32   
2    2010    1326        100             60   39   73    14    8   12  13  34   
3    2010    1393         75             43   29   60     2   15   31  14  32   
4    2010    1143         95             61   29   61     7   30   35  15  30   

   TO  Win  
0  12    1  
1  26    1  
2   6    1  
3  21    1  
4  10    1  


In [63]:
team_season_stats = team_games.groupby(['Season', 'TeamID'])[['Win', 'PointsFor', 'PointsAgainst']].agg('mean').reset_index()
team_season_stats['point_differential'] = team_season_stats['PointsFor'] - team_season_stats['PointsAgainst']
team_season_stats = team_season_stats.drop(columns={'PointsFor', 'PointsAgainst'})
team_season_stats

,Season,TeamID,Win,point_differential
0,2010,1102,0.275862,-7.344828
1,2010,1103,0.696970,4.575758
2,2010,1104,0.531250,3.718750
3,2010,1105,0.347826,-6.478261
4,2010,1106,0.464286,-2.392857
...,...,...,...,...
5999,2026,1477,0.300000,-8.966667
6000,2026,1478,0.451613,-1.580645
6001,2026,1479,0.468750,-0.406250
6002,2026,1480,0.433333,-5.166667


In [64]:
four_factors = team_games.groupby(['Season', 'TeamID'])[['FGM', 'FGM3', 'FGA', 'FTA', 'TO', 'OR', 'DR', 'FTM']].agg('sum').reset_index()
four_factors['eFG%'] = (four_factors['FGM'] + 0.5*four_factors['FGM3']) / four_factors['FGA']
four_factors['TO_rate'] = four_factors['TO'] / (four_factors['FGA'] + 0.44*four_factors['FTA'] + four_factors['TO'])
four_factors['OR_rate']= four_factors['OR'] / (four_factors['OR'] + four_factors['DR'])
four_factors['FT_rate'] = four_factors['FTM']/four_factors['FGA']
four_factors= four_factors.drop(columns={'FGM', 'FGA', 'FGM3', 'FTA', 'TO', 'OR', 'DR', 'FTM'})
four_factors

,Season,TeamID,eFG%,TO_rate,OR_rate,FT_rate
0,2010,1102,0.500378,0.195965,0.253886,0.219365
1,2010,1103,0.492639,0.169317,0.372171,0.247108
2,2010,1104,0.489688,0.165683,0.342806,0.242475
3,2010,1105,0.403802,0.186780,0.392421,0.308745
4,2010,1106,0.451936,0.201710,0.374374,0.293057
...,...,...,...,...,...,...
5999,2026,1477,0.492234,0.164856,0.253714,0.217443
6000,2026,1478,0.526520,0.163887,0.219414,0.283075
6001,2026,1479,0.494603,0.118072,0.257576,0.189423
6002,2026,1480,0.496025,0.130997,0.302817,0.205617


In [65]:
team_season_stats = team_season_stats.merge(four_factors, on=['Season', 'TeamID'], how='right')
team_season_stats

,Season,TeamID,Win,point_differential,eFG%,TO_rate,OR_rate,FT_rate
0,2010,1102,0.275862,-7.344828,0.500378,0.195965,0.253886,0.219365
1,2010,1103,0.696970,4.575758,0.492639,0.169317,0.372171,0.247108
2,2010,1104,0.531250,3.718750,0.489688,0.165683,0.342806,0.242475
3,2010,1105,0.347826,-6.478261,0.403802,0.186780,0.392421,0.308745
4,2010,1106,0.464286,-2.392857,0.451936,0.201710,0.374374,0.293057
...,...,...,...,...,...,...,...,...
5999,2026,1477,0.300000,-8.966667,0.492234,0.164856,0.253714,0.217443
6000,2026,1478,0.451613,-1.580645,0.526520,0.163887,0.219414,0.283075
6001,2026,1479,0.468750,-0.406250,0.494603,0.118072,0.257576,0.189423
6002,2026,1480,0.433333,-5.166667,0.496025,0.130997,0.302817,0.205617


In [66]:
print(team_season_stats.describe())

            Season       TeamID          Win  point_differential         eFG%  \
count  6004.000000  6004.000000  6004.000000         6004.000000  6004.000000   
mean   2018.080113  1286.445703     0.494337           -0.210589     0.498919   
std       4.908262   105.517666     0.186244            6.635642     0.031330   
min    2010.000000  1101.000000     0.000000          -33.222222     0.392276   
25%    2014.000000  1195.000000     0.360000           -4.658854     0.477871   
50%    2018.000000  1285.500000     0.500000           -0.166667     0.498693   
75%    2022.000000  1378.000000     0.625000            4.244318     0.519395   
max    2026.000000  1481.000000     1.000000           23.787879     0.610491   

           TO_rate      OR_rate      FT_rate  
count  6004.000000  6004.000000  6004.000000  
mean      0.161439     0.291197     0.245424  
std       0.020767     0.041629     0.039722  
min       0.098457     0.153110     0.126437  
25%       0.147532     0.263555    

In [67]:
print(tourney_seeds.head())
print(tourney_seeds.describe())
print(tourney_seeds.shape)

   Season  TeamID  SeedNum
0    2010    1246        1
1    2010    1452        2
2    2010    1307        3
3    2010    1458        4
4    2010    1396        5
            Season       TeamID      SeedNum
count  1085.000000  1085.000000  1085.000000
mean   2017.896774  1292.202765     8.789862
std       5.015515   104.413594     4.671308
min    2010.000000  1101.000000     1.000000
25%    2014.000000  1211.000000     5.000000
50%    2018.000000  1287.000000     9.000000
75%    2023.000000  1388.000000    13.000000
max    2026.000000  1474.000000    16.000000
(1085, 3)


In [68]:
team_season_stats = team_season_stats.merge(tourney_seeds, on=['Season', 'TeamID'], how='inner')
team_season_stats

,Season,TeamID,Win,point_differential,eFG%,TO_rate,OR_rate,FT_rate,SeedNum
0,2010,1115,0.531250,-0.281250,0.450030,0.209554,0.329518,0.325996,16
1,2010,1124,0.774194,10.161290,0.546238,0.176383,0.315526,0.273406,3
2,2010,1139,0.875000,10.062500,0.519925,0.169655,0.281637,0.356787,5
3,2010,1140,0.848485,16.939394,0.550926,0.141078,0.275410,0.300412,7
4,2010,1143,0.696970,9.575758,0.533696,0.150928,0.321186,0.266459,8
...,...,...,...,...,...,...,...,...,...
1080,2026,1438,0.852941,12.235294,0.546161,0.129606,0.314859,0.240039,3
1081,2026,1458,0.705882,7.088235,0.544634,0.107408,0.273394,0.250712,5
1082,2026,1460,0.656250,4.562500,0.544212,0.137428,0.291583,0.282958,14
1083,2026,1465,0.757576,5.454545,0.485948,0.143177,0.331929,0.259581,13


In [69]:
print(team_season_stats.columns.tolist())

['Season', 'TeamID', 'Win', 'point_differential', 'eFG%', 'TO_rate', 'OR_rate', 'FT_rate', 'SeedNum']


In [70]:
print(massey_wide.shape)
print(massey_wide.head())
print(massey_wide.describe())

(5651, 4)
SystemName  Season  TeamID  MOR_Rank  POM_Rank
0             2010    1102     227.0     246.0
1             2010    1103     118.0     119.0
2             2010    1104      57.0      64.0
3             2010    1105     337.0     336.0
4             2010    1106     303.0     302.0
SystemName       Season       TeamID     MOR_Rank     POM_Rank
count       5651.000000  5651.000000  5651.000000  5651.000000
mean        2017.960184  1286.528225   177.158025   177.158025
std            5.035028   105.558741   102.077127   102.077127
min         2010.000000  1101.000000     1.000000     1.000000
25%         2014.000000  1195.000000    89.000000    89.000000
50%         2018.000000  1286.000000   177.000000   177.000000
75%         2023.000000  1378.000000   265.000000   265.000000
max         2026.000000  1481.000000   365.000000   365.000000


In [71]:
tournament_teams = team_season_stats.merge(massey_wide, on=['Season', 'TeamID'], how='left')
print(tournament_teams.shape)
print(tournament_teams.isnull().sum())
print(tournament_teams.head())

(1085, 11)
Season                0
TeamID                0
Win                   0
point_differential    0
eFG%                  0
TO_rate               0
OR_rate               0
FT_rate               0
SeedNum               0
MOR_Rank              0
POM_Rank              0
dtype: int64
   Season  TeamID       Win  point_differential      eFG%   TO_rate   OR_rate  \
0    2010    1115  0.531250           -0.281250  0.450030  0.209554  0.329518   
1    2010    1124  0.774194           10.161290  0.546238  0.176383  0.315526   
2    2010    1139  0.875000           10.062500  0.519925  0.169655  0.281637   
3    2010    1140  0.848485           16.939394  0.550926  0.141078  0.275410   
4    2010    1143  0.696970            9.575758  0.533696  0.150928  0.321186   

    FT_rate  SeedNum  MOR_Rank  POM_Rank  
0  0.325996       16     269.0     238.0  
1  0.273406        3      11.0      12.0  
2  0.356787        5      24.0      26.0  
3  0.300412        7      14.0       7.0  
4  0.26645

In [74]:
tournament_teams = tournament_teams.sort_values(by=['TeamID'], ascending=[True])
tournament_teams

,Season,TeamID,Win,point_differential,eFG%,TO_rate,OR_rate,FT_rate,SeedNum,MOR_Rank,POM_Rank
609,2019,1101,0.793103,6.827586,0.524345,0.154810,0.283711,0.249688,15,169.0,145.0
677,2021,1101,0.826087,14.565217,0.529087,0.165216,0.296247,0.234168,14,77.0,86.0
65,2011,1103,0.647059,3.647059,0.498715,0.155703,0.293562,0.225707,15,110.0,123.0
201,2013,1103,0.806452,9.677419,0.513598,0.172013,0.344710,0.236261,12,60.0,54.0
745,2022,1103,0.709677,5.129032,0.527933,0.150589,0.251029,0.277467,13,75.0,131.0
...,...,...,...,...,...,...,...,...,...,...,...
948,2024,1463,0.689655,5.413793,0.522885,0.117504,0.246862,0.190035,13,98.0,84.0
1015,2025,1463,0.750000,10.428571,0.547535,0.119552,0.261445,0.235915,13,68.0,73.0
1083,2026,1465,0.757576,5.454545,0.485948,0.143177,0.331929,0.259581,13,114.0,106.0
1016,2025,1471,0.875000,15.093750,0.554283,0.111797,0.234661,0.252046,12,45.0,37.0


In [73]:
tourney_results

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT
0,2010,134,1115,61,1457,44,N,0
1,2010,136,1124,68,1358,59,N,0
2,2010,136,1139,77,1431,59,N,0
3,2010,136,1140,99,1196,92,N,2
4,2010,136,1242,90,1250,74,N,0
...,...,...,...,...,...,...,...,...
996,2025,146,1120,70,1277,64,N,0
997,2025,146,1222,69,1397,50,N,0
998,2025,152,1196,79,1120,73,N,0
999,2025,152,1222,70,1181,67,N,0


In [111]:
matchups = tourney_results.copy()
matchups['TeamID_Low'] = matchups[['WTeamID', 'LTeamID']].min(axis=1)
matchups['TeamID_High'] = matchups[['WTeamID', 'LTeamID']].max(axis=1)
matchups['Label'] = (matchups['WTeamID']==matchups['TeamID_Low']).astype(int)
matchups = matchups.drop(columns={'WTeamID', 'LTeamID', 'LScore', 'WLoc', 'WScore', 'NumOT', 'DayNum'})
matchups

,Season,TeamID_Low,TeamID_High,Label
0,2010,1115,1457,1
1,2010,1124,1358,1
2,2010,1139,1431,1
3,2010,1140,1196,1
4,2010,1242,1250,1
...,...,...,...,...
996,2025,1120,1277,1
997,2025,1222,1397,1
998,2025,1120,1196,0
999,2025,1181,1222,0


In [112]:
matchups = matchups.merge(tournament_teams, left_on=['TeamID_Low', 'Season'], right_on=['TeamID', 'Season'])
matchups

,Season,TeamID_Low,TeamID_High,Label,TeamID,Win,point_differential,eFG%,TO_rate,OR_rate,FT_rate,SeedNum,MOR_Rank,POM_Rank
0,2010,1115,1457,1,1115,0.531250,-0.281250,0.450030,0.209554,0.329518,0.325996,16,269.0,238.0
1,2010,1124,1358,1,1124,0.774194,10.161290,0.546238,0.176383,0.315526,0.273406,3,11.0,12.0
2,2010,1139,1431,1,1139,0.875000,10.062500,0.519925,0.169655,0.281637,0.356787,5,24.0,26.0
3,2010,1140,1196,1,1140,0.848485,16.939394,0.550926,0.141078,0.275410,0.300412,7,14.0,7.0
4,2010,1242,1250,1,1242,0.941176,17.970588,0.550553,0.157505,0.319303,0.297284,1,1.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
996,2025,1120,1277,1,1120,0.848485,14.242424,0.557115,0.112015,0.298401,0.248154,1,4.0,4.0
997,2025,1222,1397,1,1222,0.882353,15.735294,0.526763,0.116411,0.327338,0.208604,1,3.0,3.0
998,2025,1120,1196,0,1120,0.848485,14.242424,0.557115,0.112015,0.298401,0.248154,1,4.0,4.0
999,2025,1181,1222,0,1181,0.911765,20.794118,0.573632,0.120071,0.277049,0.251741,1,1.0,1.0


In [113]:
matchups = matchups.drop(columns={'TeamID'})
matchups = matchups.rename(columns={
    'Win' : 'Win_low',
    'point_differential': 'point_differential_low',
    'eFG%': 'eFG%_low',
    'TO_rate': 'TO_rate_low',
    'OR_rate': 'OR_rate_low',
    'FT_rate': 'FT_rate_low',
    'SeedNum': 'SeedNum_low',
    'MOR_Rank': 'MOR_Rank_low',
    'POM_Rank': 'POM_Rank_low'
})
print(matchups.columns.tolist())


['Season', 'TeamID_Low', 'TeamID_High', 'Label', 'Win_low', 'point_differential_low', 'eFG%_low', 'TO_rate_low', 'OR_rate_low', 'FT_rate_low', 'SeedNum_low', 'MOR_Rank_low', 'POM_Rank_low']


In [114]:
matchups.head()

,Season,TeamID_Low,TeamID_High,Label,Win_low,point_differential_low,eFG%_low,TO_rate_low,OR_rate_low,FT_rate_low,SeedNum_low,MOR_Rank_low,POM_Rank_low
0,2010,1115,1457,1,0.531250,-0.281250,0.450030,0.209554,0.329518,0.325996,16,269.0,238.0
1,2010,1124,1358,1,0.774194,10.161290,0.546238,0.176383,0.315526,0.273406,3,11.0,12.0
2,2010,1139,1431,1,0.875000,10.062500,0.519925,0.169655,0.281637,0.356787,5,24.0,26.0
3,2010,1140,1196,1,0.848485,16.939394,0.550926,0.141078,0.275410,0.300412,7,14.0,7.0
4,2010,1242,1250,1,0.941176,17.970588,0.550553,0.157505,0.319303,0.297284,1,1.0,2.0


In [115]:
matchups = matchups.merge(tournament_teams, left_on=['TeamID_High', 'Season'], right_on=['TeamID', 'Season'])
matchups = matchups.drop(columns={'TeamID'})
matchups = matchups.rename(columns={
    'Win' : 'Win_high',
    'point_differential': 'point_differential_high',
    'eFG%': 'eFG%_high',
    'TO_rate': 'TO_rate_high',
    'OR_rate': 'OR_rate_high',
    'FT_rate': 'FT_rate_high',
    'SeedNum': 'SeedNum_high',
    'MOR_Rank': 'MOR_Rank_high',
    'POM_Rank': 'POM_Rank_high'
})
matchups.head()

,Season,TeamID_Low,TeamID_High,Label,Win_low,point_differential_low,eFG%_low,TO_rate_low,OR_rate_low,FT_rate_low,...,POM_Rank_low,Win_high,point_differential_high,eFG%_high,TO_rate_high,OR_rate_high,FT_rate_high,SeedNum_high,MOR_Rank_high,POM_Rank_high
0,2010,1115,1457,1,0.531250,-0.281250,0.450030,0.209554,0.329518,0.325996,...,238.0,0.566667,0.633333,0.418934,0.152211,0.353606,0.226190,16,199.0,212.0
1,2010,1124,1358,1,0.774194,10.161290,0.546238,0.176383,0.315526,0.273406,...,12.0,0.750000,7.571429,0.543372,0.163675,0.330392,0.279780,14,112.0,102.0
2,2010,1139,1431,1,0.875000,10.062500,0.519925,0.169655,0.281637,0.356787,...,26.0,0.812500,11.593750,0.529912,0.164043,0.272571,0.271131,12,38.0,34.0
3,2010,1140,1196,1,0.848485,16.939394,0.550926,0.141078,0.275410,0.300412,...,7.0,0.636364,6.181818,0.494035,0.153618,0.356557,0.237552,10,40.0,49.0
4,2010,1242,1250,1,0.941176,17.970588,0.550553,0.157505,0.319303,0.297284,...,2.0,0.687500,5.093750,0.515634,0.163181,0.279110,0.310999,16,149.0,180.0


In [116]:
print(matchups.columns.tolist())

['Season', 'TeamID_Low', 'TeamID_High', 'Label', 'Win_low', 'point_differential_low', 'eFG%_low', 'TO_rate_low', 'OR_rate_low', 'FT_rate_low', 'SeedNum_low', 'MOR_Rank_low', 'POM_Rank_low', 'Win_high', 'point_differential_high', 'eFG%_high', 'TO_rate_high', 'OR_rate_high', 'FT_rate_high', 'SeedNum_high', 'MOR_Rank_high', 'POM_Rank_high']


In [118]:
matchups['win_diff'] = matchups['Win_low'] - matchups['Win_high']
matchups['point_differential_diff'] = matchups['point_differential_low'] - matchups['point_differential_high']
matchups['eFG%_diff'] = matchups['eFG%_low'] - matchups['eFG%_high']
matchups['TO_rate_diff'] = matchups['TO_rate_low'] - matchups['TO_rate_high']
matchups['OR_rate_diff'] = matchups['OR_rate_low'] - matchups['OR_rate_high']
matchups['FT_rate_diff'] = matchups['FT_rate_low'] - matchups['FT_rate_high']
matchups['SeedNum_diff'] = matchups['SeedNum_low'] - matchups['SeedNum_high']
matchups['MOR_Rank_diff'] = matchups['MOR_Rank_low'] - matchups['MOR_Rank_high']
matchups['POM_Rank_diff'] = matchups['POM_Rank_low'] - matchups['POM_Rank_high']

# drop cols
matchups = matchups.drop(columns={
    'Win_low', 'point_differential_low',
    'eFG%_low', 'TO_rate_low', 'OR_rate_low',
    'FT_rate_low', 'SeedNum_low', 'MOR_Rank_low',
    'POM_Rank_low', 'Win_high', 'point_differential_high',
    'eFG%_high', 'TO_rate_high', 'OR_rate_high', 'FT_rate_high', 
    'SeedNum_high', 'MOR_Rank_high', 'POM_Rank_high'
})

print(matchups.shape)
print(matchups.columns.tolist())
print(matchups.head())

(1001, 13)
['Season', 'TeamID_Low', 'TeamID_High', 'Label', 'win_diff', 'point_differential_diff', 'eFG%_diff', 'TO_rate_diff', 'OR_rate_diff', 'FT_rate_diff', 'SeedNum_diff', 'MOR_Rank_diff', 'POM_Rank_diff']
   Season  TeamID_Low  TeamID_High  Label  win_diff  point_differential_diff  \
0    2010        1115         1457      1 -0.035417                -0.914583   
1    2010        1124         1358      1  0.024194                 2.589862   
2    2010        1139         1431      1  0.062500                -1.531250   
3    2010        1140         1196      1  0.212121                10.757576   
4    2010        1242         1250      1  0.253676                12.876838   

   eFG%_diff  TO_rate_diff  OR_rate_diff  FT_rate_diff  SeedNum_diff  \
0   0.031096      0.057344     -0.024088      0.099806             0   
1   0.002866      0.012708     -0.014866     -0.006374           -11   
2  -0.009987      0.005612      0.009066      0.085656            -7   
3   0.056891     -0.0

In [119]:
print(matchups.shape)
print(matchups.isnull().sum())
print(matchups['Label'].value_counts())

(1001, 13)
Season                     0
TeamID_Low                 0
TeamID_High                0
Label                      0
win_diff                   0
point_differential_diff    0
eFG%_diff                  0
TO_rate_diff               0
OR_rate_diff               0
FT_rate_diff               0
SeedNum_diff               0
MOR_Rank_diff              0
POM_Rank_diff              0
dtype: int64
Label
1    515
0    486
Name: count, dtype: int64


In [121]:
matchups['Season'].unique()

array([2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2021,
       2022, 2023, 2024, 2025])

In [ ]:
X = matchups[['win_diff', 'point_differential_diff', 'eFG%_diff', 'TO_rate_diff', 'OR_rate_diff', 'FT_rate_diff', 'SeedNum_diff', 'MOR_Rank_diff', 'POM_Rank_diff']]
y = matchups['Label']

X_train = X[matchups['Season'] <= 2024]
X_test  = X[matchups['Season'] == 2025]
y_train = y[matchups['Season'] <= 2024]
y_test  = y[matchups['Season'] == 2025]


In [125]:
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

X_train: (934, 9)
X_test: (67, 9)
y_train: (934,)
y_test: (67,)


In [126]:
X_train.describe()

,win_diff,point_differential_diff,eFG%_diff,TO_rate_diff,OR_rate_diff,FT_rate_diff,SeedNum_diff,MOR_Rank_diff,POM_Rank_diff
count,934.000000,934.000000,934.000000,934.000000,934.000000,934.000000,934.000000,934.000000,934.000000
mean,0.004466,0.538719,0.006523,0.001881,-0.000571,0.000775,-0.328694,-2.076017,-1.658458
std,0.146292,6.583144,0.038996,0.022356,0.050703,0.046245,7.480599,72.255869,71.067108
min,-0.633333,-21.789661,-0.143802,-0.067731,-0.196924,-0.190744,-15.000000,-266.000000,-284.000000
25%,-0.086174,-3.733695,-0.018959,-0.013234,-0.032780,-0.029706,-7.000000,-30.000000,-30.000000
50%,0.002042,0.250000,0.006536,0.001936,-0.000575,-0.000367,0.000000,-2.000000,-1.000000
75%,0.097309,4.741951,0.032965,0.016938,0.034939,0.031109,5.000000,25.000000,25.000000
max,0.486631,24.486742,0.113413,0.093189,0.134435,0.150791,15.000000,292.000000,305.000000


In [131]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression()

model.fit(X_train_scaled, y_train)
pred = model.predict_proba(X_test_scaled)[:,1]

brier = brier_score_loss(y_test, pred)
print(f"Brier Score: {brier:.4f}")

Brier Score: 0.1577
